In [0]:
-- 1. Read, Type, Trim, Deduplicate, and Derive Calendar Attributes
-- Using TEMPORARY VIEW because global DISTINCT is best handled in batch for reference/calendar data
CREATE TEMPORARY VIEW stg_calendar_typed AS
SELECT DISTINCT
  CAST(TRIM(calendar_date) AS DATE) AS calendar_date,
  
  -- Dynamically generate the missing calendar columns
  YEAR(CAST(TRIM(calendar_date) AS DATE)) AS year,
  MONTH(CAST(TRIM(calendar_date) AS DATE)) AS month,
  DAY(CAST(TRIM(calendar_date) AS DATE)) AS day,
  QUARTER(CAST(TRIM(calendar_date) AS DATE)) AS quarter,
  DAYOFWEEK(CAST(TRIM(calendar_date) AS DATE)) AS day_of_week, 
  date_format(CAST(TRIM(calendar_date) AS DATE), 'EEEE') AS day_name,
  date_format(CAST(TRIM(calendar_date) AS DATE), 'MMMM') AS month_name,
  date_format(CAST(TRIM(calendar_date) AS DATE), 'yyyy-MM') AS year_month,
  CASE WHEN DAYOFWEEK(CAST(TRIM(calendar_date) AS DATE)) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  
  _ingestion_timestamp,
  _source_system
FROM mavenmarket.bronze.calendar;

-- 2 & 3. Run KPI Checks & Consolidate Flags
CREATE TEMPORARY VIEW stg_calendar_flagged AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN calendar_date IS NULL THEN 'Completeness: calendar_date is null; ' ELSE '' END AS hard_failures,
  '' AS dq_flags -- No warn checks needed for standard calendar
FROM stg_calendar_typed;

-- 4. Write Target Tables (Split the rows)
-- Replaced LIVE TABLE with MATERIALIZED VIEW for the latest syntax
CREATE MATERIALIZED VIEW silver_calendar
COMMENT "Cleaned and enriched calendar dimension"
AS SELECT 
  * EXCEPT(hard_failures, dq_flags) 
FROM stg_calendar_flagged 
WHERE hard_failures = '';

CREATE MATERIALIZED VIEW quarantine_calendar
COMMENT "Quarantined calendar dates"
AS SELECT 
  * EXCEPT(dq_flags), 
  hard_failures AS quarantine_reasons 
FROM stg_calendar_flagged 
WHERE hard_failures != '';